# Step 3: PubMed Data Preprocessing

This notebook handles the preprocessing of PubMed research data:
1. Load Preprocessed_pubmed_data.xls
2. Drop rows with missing critical data
3. Normalize text and fill missing values
4. Save cleaned CSV

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import os

## 1. Load PubMed Dataset

In [8]:
# Define file paths
PUBMED_DATA_PATH = '../data/pre-processed/pubmed/'
OUTPUT_PATH = '../data/pre-processed/pubmed/'

# Ensure output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load PubMed preprocessed data
# Note: The .xls file is actually a CSV file (comma-separated) despite the extension
input_file = os.path.join(PUBMED_DATA_PATH, 'Preprocessed_pubmed_data.xls')
print(f"Loading PubMed data from {input_file}...")

# Read as CSV with comma separator
pubmed_df = pd.read_csv(input_file, sep=',', on_bad_lines='warn', low_memory=False)
print(f"PubMed dataset loaded successfully!")

print(f"\nPubMed dataset shape: {pubmed_df.shape}")

Loading PubMed data from ../data/pre-processed/pubmed/Preprocessed_pubmed_data.xls...
PubMed dataset loaded successfully!

PubMed dataset shape: (30567, 11)


In [9]:
# Inspect the dataset
print("PubMed Dataset Columns:")
print(pubmed_df.columns.tolist())
print("\nFirst 5 rows:")
pubmed_df.head()

PubMed Dataset Columns:
['ingredient', 'pmid', 'title', 'year', 'journal', 'diseases_mentioned', 'mesh_disease_terms', 'relationship_type', 'key_findings', 'authors', 'pubmed_url']

First 5 rows:


,ingredient,pmid,title,year,journal,diseases_mentioned,mesh_disease_terms,relationship_type,key_findings,authors,pubmed_url
0,protein,41390803,Association between triglyceride-glucose index...,2025,Cardiovascular diabetology,"diabetes, insulin resistance, cardiovascular d...",NaN,Risk Factor,The results indicated that elevated TyG-ABSI v...,"Xinyi Shao, Zhaofu Tan, Lifu Sun",https://pubmed.ncbi.nlm.nih.gov/41390803/
1,protein,41390778,Multi-task learning identifies shared genetic ...,2025,Scientific reports,alzheimer,NaN,Risk Factor,We analyzed electronic health records from UCL...,"Mingzhou Fu, Thai Tran, Bogdan Pasaniuc",https://pubmed.ncbi.nlm.nih.gov/41390778/
2,protein,41390701,Obesity concurrent with gestational diabetes m...,2025,Scientific reports,"diabetes, obesity, diabetes mellitus",NaN,Risk Factor,"Of note, proximity ligation assay (PLA) reveal...","Ruofan Wang, Jianying Liu, Qian Li",https://pubmed.ncbi.nlm.nih.gov/41390701/
3,protein,41390575,The RNA-binding protein CPEB1 marks healthy ad...,2025,Scientific reports,"diabetes, type 2 diabetes",NaN,Associated/Neutral,See full abstract for details,"Sutichot D Nimkulrat, Matthew R Wagner, Bayley...",https://pubmed.ncbi.nlm.nih.gov/41390575/
4,protein,41390442,Genetic insights into 5-LOX-activating protein...,2025,Human genomics,"inflammation, cardiovascular disease, stroke, ...",NaN,Associated/Neutral,The 5-lipoxygenase-activating protein (FLAP) i...,"Katharina Rataj, Ulrike Garscha",https://pubmed.ncbi.nlm.nih.gov/41390442/


In [10]:
# Check data types
print("Data types:")
print(pubmed_df.dtypes)

Data types:
ingredient            object
pmid                   int64
title                 object
year                   int64
journal               object
diseases_mentioned    object
mesh_disease_terms    object
relationship_type     object
key_findings          object
authors               object
pubmed_url            object
dtype: object


## 2. Explore Data Quality

In [11]:
# Check missing values
print("Missing values per column:")
missing_counts = pubmed_df.isnull().sum()
missing_pct = (pubmed_df.isnull().sum() / len(pubmed_df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_pct.round(2)
})
print(missing_df)

Missing values per column:
                    Missing Count  Missing Percentage
ingredient                      0                0.00
pmid                            0                0.00
title                           0                0.00
year                            0                0.00
journal                         0                0.00
diseases_mentioned              0                0.00
mesh_disease_terms          18375               60.11
relationship_type               0                0.00
key_findings                    0                0.00
authors                         0                0.00
pubmed_url                      0                0.00


In [12]:
# Check sample values for each column
print("Sample values for each column:")
for col in pubmed_df.columns:
    print(f"\n--- {col} ---")
    print(pubmed_df[col].head(3).tolist())

Sample values for each column:

--- ingredient ---
['protein', 'protein', 'protein']

--- pmid ---
[41390803, 41390778, 41390701]

--- title ---
['Association between triglyceride-glucose index combined with a body shape index and atherosclerotic cardiovascular disease risk varies by glycemic status: insights from the UK Biobank.', "Multi-task learning identifies shared genetic risk for late-onset epilepsy and alzheimer's disease.", 'Obesity concurrent with gestational diabetes mellitus dysregulates mitochondria-endoplasmic reticulum contacts in human placenta.']

--- year ---
[2025, 2025, 2025]

--- journal ---
['Cardiovascular diabetology', 'Scientific reports', 'Scientific reports']

--- diseases_mentioned ---
['diabetes, insulin resistance, cardiovascular disease', 'alzheimer', 'diabetes, obesity, diabetes mellitus']

--- mesh_disease_terms ---
[nan, nan, nan]

--- relationship_type ---
['Risk Factor', 'Risk Factor', 'Risk Factor']

--- key_findings ---
['The results indicated that

## 3. Drop Rows with Missing Critical Data

In [13]:
# Define critical columns (adjust based on your data)
# Common critical columns for PubMed data: title, authors, diseases_mentioned, ingredient
critical_columns = ['title', 'authors', 'diseases_mentioned']

# Check which critical columns exist in the dataset
existing_critical = [col for col in critical_columns if col in pubmed_df.columns]
print(f"Critical columns found: {existing_critical}")

# Count rows before dropping
initial_count = len(pubmed_df)
print(f"\nTotal rows before dropping: {initial_count}")

# Drop rows with missing critical data
if existing_critical:
    pubmed_df = pubmed_df.dropna(subset=existing_critical)
    dropped_count = initial_count - len(pubmed_df)
    print(f"Rows dropped due to missing critical data: {dropped_count}")
    print(f"Rows remaining: {len(pubmed_df)}")
else:
    print("No critical columns found. Skipping drop step.")

Critical columns found: ['title', 'authors', 'diseases_mentioned']

Total rows before dropping: 30567
Rows dropped due to missing critical data: 0
Rows remaining: 30567


In [14]:
# Verify critical columns are now complete
print("Missing values after dropping rows:")
print(pubmed_df.isnull().sum())

Missing values after dropping rows:
ingredient                0
pmid                      0
title                     0
year                      0
journal                   0
diseases_mentioned        0
mesh_disease_terms    18375
relationship_type         0
key_findings              0
authors                   0
pubmed_url                0
dtype: int64


## 4. Normalize Text

In [15]:
def normalize_text(text):
    """
    Normalize text by:
    - Converting to string
    - Removing extra whitespace
    - Stripping leading/trailing whitespace
    - Standardizing common patterns
    """
    if pd.isna(text):
        return ''
    
    text = str(text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

# Apply normalization to all string columns
string_columns = pubmed_df.select_dtypes(include=['object']).columns

print(f"Normalizing {len(string_columns)} text columns...")
for col in string_columns:
    pubmed_df[col] = pubmed_df[col].apply(normalize_text)

print("Text normalization completed!")

Normalizing 9 text columns...
Text normalization completed!


In [16]:
# Normalize specific columns if they exist
# Convert ingredient names to lowercase for consistency
if 'ingredient' in pubmed_df.columns:
    pubmed_df['ingredient'] = pubmed_df['ingredient'].str.lower()
    print("Ingredient column normalized to lowercase.")
    print(f"Unique ingredients: {pubmed_df['ingredient'].nunique()}")

# Convert disease names to lowercase
if 'diseases_mentioned' in pubmed_df.columns:
    pubmed_df['diseases_mentioned'] = pubmed_df['diseases_mentioned'].str.lower()
    print("Diseases column normalized to lowercase.")

Ingredient column normalized to lowercase.
Unique ingredients: 224
Diseases column normalized to lowercase.


## 5. Fill Missing Values

In [17]:
# Define fill values for non-critical columns
fill_values = {
    'mesh_disease_terms': '',
    'key_findings': 'No findings available',
    'pubmed_url': '',
    'journal': 'Unknown Journal',
    'relationship_type': 'Unknown'
}

# Fill missing values
print("Filling missing values...")
for col, fill_val in fill_values.items():
    if col in pubmed_df.columns:
        before_null = pubmed_df[col].isnull().sum()
        pubmed_df[col] = pubmed_df[col].fillna(fill_val)
        after_null = pubmed_df[col].isnull().sum()
        print(f"  {col}: {before_null} → {after_null} null values")

# Fill any remaining text columns with empty string
text_cols = pubmed_df.select_dtypes(include=['object']).columns
for col in text_cols:
    pubmed_df[col] = pubmed_df[col].fillna('')

# Fill numeric columns with appropriate defaults
numeric_cols = pubmed_df.select_dtypes(include=['number']).columns
for col in numeric_cols:
    if 'year' in col.lower():
        pubmed_df[col] = pubmed_df[col].fillna(0).astype(int)
    else:
        pubmed_df[col] = pubmed_df[col].fillna(0)

print("\nMissing values after filling:")
print(pubmed_df.isnull().sum())

Filling missing values...
  mesh_disease_terms: 0 → 0 null values
  key_findings: 0 → 0 null values
  pubmed_url: 0 → 0 null values
  journal: 0 → 0 null values
  relationship_type: 0 → 0 null values

Missing values after filling:
ingredient            0
pmid                  0
title                 0
year                  0
journal               0
diseases_mentioned    0
mesh_disease_terms    0
relationship_type     0
key_findings          0
authors               0
pubmed_url            0
dtype: int64


## 6. Remove Duplicates

In [18]:
# Check for duplicates
print(f"Total rows before deduplication: {len(pubmed_df)}")

# Check duplicates based on key columns (e.g., pmid if exists)
if 'pmid' in pubmed_df.columns:
    duplicates = pubmed_df.duplicated(subset=['pmid']).sum()
    print(f"Duplicate pmid entries: {duplicates}")
    pubmed_df = pubmed_df.drop_duplicates(subset=['pmid'])
else:
    duplicates = pubmed_df.duplicated().sum()
    print(f"Duplicate rows: {duplicates}")
    pubmed_df = pubmed_df.drop_duplicates()

print(f"Total rows after deduplication: {len(pubmed_df)}")

Total rows before deduplication: 30567
Duplicate pmid entries: 6008
Total rows after deduplication: 24559


## 7. Data Validation

In [19]:
# Final data quality check
print("Final Dataset Summary:")
print(f"Total rows: {len(pubmed_df)}")
print(f"Total columns: {len(pubmed_df.columns)}")
print("\nData types:")
print(pubmed_df.dtypes)

Final Dataset Summary:
Total rows: 24559
Total columns: 11

Data types:
ingredient            object
pmid                   int64
title                 object
year                   int64
journal               object
diseases_mentioned    object
mesh_disease_terms    object
relationship_type     object
key_findings          object
authors               object
pubmed_url            object
dtype: object


In [20]:
# Check distribution of key columns
print("\nColumn value distributions:")

if 'relationship_type' in pubmed_df.columns:
    print("\nRelationship Types:")
    print(pubmed_df['relationship_type'].value_counts())

if 'ingredient' in pubmed_df.columns:
    print("\nTop 10 Ingredients:")
    print(pubmed_df['ingredient'].value_counts().head(10))

if 'year' in pubmed_df.columns:
    print("\nYear Distribution:")
    print(pubmed_df['year'].value_counts().sort_index().tail(10))


Column value distributions:

Relationship Types:
relationship_type
Protective/Beneficial    8410
Associated/Neutral       8107
Risk Factor              8042
Name: count, dtype: int64

Top 10 Ingredients:
ingredient
total fat                   176
protein                     165
saturated fatty acids       165
citrus fruit                165
high fructose corn syrup    164
cholesterol                 162
canola oil                  160
refined grain               160
omega-6 fatty acids         159
vitamin a                   158
Name: count, dtype: int64

Year Distribution:
year
2017      228
2018      291
2019      437
2020      641
2021      995
2022     1352
2023     1930
2024     3256
2025    13756
2026      479
Name: count, dtype: int64


In [21]:
# Sample of cleaned data
print("\nSample of cleaned data:")
pubmed_df.head()


Sample of cleaned data:


,ingredient,pmid,title,year,journal,diseases_mentioned,mesh_disease_terms,relationship_type,key_findings,authors,pubmed_url
0,protein,41390803,Association between triglyceride-glucose index...,2025,Cardiovascular diabetology,"diabetes, insulin resistance, cardiovascular d...",,Risk Factor,The results indicated that elevated TyG-ABSI v...,"Xinyi Shao, Zhaofu Tan, Lifu Sun",https://pubmed.ncbi.nlm.nih.gov/41390803/
1,protein,41390778,Multi-task learning identifies shared genetic ...,2025,Scientific reports,alzheimer,,Risk Factor,We analyzed electronic health records from UCL...,"Mingzhou Fu, Thai Tran, Bogdan Pasaniuc",https://pubmed.ncbi.nlm.nih.gov/41390778/
2,protein,41390701,Obesity concurrent with gestational diabetes m...,2025,Scientific reports,"diabetes, obesity, diabetes mellitus",,Risk Factor,"Of note, proximity ligation assay (PLA) reveal...","Ruofan Wang, Jianying Liu, Qian Li",https://pubmed.ncbi.nlm.nih.gov/41390701/
3,protein,41390575,The RNA-binding protein CPEB1 marks healthy ad...,2025,Scientific reports,"diabetes, type 2 diabetes",,Associated/Neutral,See full abstract for details,"Sutichot D Nimkulrat, Matthew R Wagner, Bayley...",https://pubmed.ncbi.nlm.nih.gov/41390575/
4,protein,41390442,Genetic insights into 5-LOX-activating protein...,2025,Human genomics,"inflammation, cardiovascular disease, stroke, ...",,Associated/Neutral,The 5-lipoxygenase-activating protein (FLAP) i...,"Katharina Rataj, Ulrike Garscha",https://pubmed.ncbi.nlm.nih.gov/41390442/


## 8. Save Cleaned Dataset

In [22]:
# Save the cleaned dataset
output_file = os.path.join(OUTPUT_PATH, 'cleaned_pubmed_data.csv')

print(f"Saving cleaned data to {output_file}...")
pubmed_df.to_csv(output_file, index=False)

# Verify file was saved
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file) / (1024 * 1024)  # Convert to MB
    print(f"\nFile saved successfully!")
    print(f"File size: {file_size:.2f} MB")
else:
    print("Error: File was not saved!")

Saving cleaned data to ../data/pre-processed/pubmed/cleaned_pubmed_data.csv...

File saved successfully!
File size: 10.55 MB


In [23]:
# Summary
print("\n" + "="*50)
print("PubMed Data Preprocessing Complete!")
print("="*50)
print(f"\nInput file: Preprocessed_pubmed_data.xls")
print(f"  - Original records: {initial_count}")
print(f"\nOutput file: cleaned_pubmed_data.csv")
print(f"  - Cleaned records: {len(pubmed_df)}")
print(f"  - Columns: {len(pubmed_df.columns)}")
print(f"\nKey statistics:")
if 'ingredient' in pubmed_df.columns:
    print(f"  - Unique ingredients: {pubmed_df['ingredient'].nunique()}")
if 'pmid' in pubmed_df.columns:
    print(f"  - Unique publications: {pubmed_df['pmid'].nunique()}")


PubMed Data Preprocessing Complete!

Input file: Preprocessed_pubmed_data.xls
  - Original records: 30567

Output file: cleaned_pubmed_data.csv
  - Cleaned records: 24559
  - Columns: 11

Key statistics:
  - Unique ingredients: 221
  - Unique publications: 24559
